<a href="https://colab.research.google.com/github/sakshammverma/finetuing_llm/blob/main/unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# xformers --> memory efficient attention kernels

In [ ]:
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import torch
import random
import numpy as np

In [ ]:
# setting up seed for python, numpy, PyTorch, and for GPU's for reproducibility of random weights init and all
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

In [ ]:
# all this code is abt performance optimization and memory optimization
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')


max_seq_len = 4096 #attention span of my model
dtype= None #placeholde, letting the system optimally detect
load_in_4bit = True #quantization (qLoRA)

In [ ]:
assert torch.cuda.is_available()

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
base_model_name = 'unsloth/tinyllama-bnb-4bit'
model, tokenizer  = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_len,
    dtype = dtype,
    load_in_4bit=True
)

In [ ]:
model

In [ ]:
tokenizer

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
     r= 32, #lora rank (8/16/32/64)
    target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
],
    lora_alpha = 64, #usually r or 2*r
    lora_dropout=0.0, #unsloth recommends 0
    bias = 'none', #good practice
    use_gradient_checkpointing=False, #set tRUE if model > 7B
    random_state = 3407
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
model.peft_config

In [ ]:
import torch
torch.cuda.memory_summary

In [ ]:
eos_token = tokenizer.eos_token

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + eos_token
        texts.append(text)
    return { "text" : texts}

In [ ]:
from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")

dataset = dataset.shuffle(seed=3407).select(range(1500))
# Apply the formatting function
dataset = dataset.map(formatting_prompts_func,
                      batched = True,
                      remove_columns=dataset.column_names #keeps only ttext
                      )

In [ ]:
dataset

In [ ]:
dataset[0]

In [ ]:
!pip install psutil

In [ ]:
import psutil, time #lib to monitor system ram
torch.cuda.empty_cache() #release unused cache memory held by pytorch
torch.cuda.reset_peak_memory_stats() #allows to measure maximum vram consumed from this point forward

In [ ]:
# before stats of fine tuning for performance benchmarking
process = psutil.Process() #current python interpretor process
train_start_time = time.time() #exact timestamp in sec
cpu_ram_before = process.memory_info().rss / 1024**3 #how much system ran (not GPU) is used before model loads data batches    (rss-> Resident Set Size in GB's)


In [ ]:
process

In [ ]:
train_start_time

In [ ]:
cpu_ram_before

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_len,
    dataset_num_proc = 2,
    packing = True, #packs multiple short seq in 1 sequence
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Set this to a higher number for a full run (e.g., 300)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = 'none',
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
#  GPU Memory Stats
used_vram = round(torch.cuda.max_memory_reserved() / 1024**3, 3) # GB
used_memory_stats = torch.cuda.get_device_properties(0).total_memory / 1.074e+9
used_memory_percentage = round(used_vram / used_memory_stats * 100, 3)

# End timers and CPU RAM tracking
train_end_time = time.time()
train_duration = train_end_time - train_start_time # in seconds
cpu_ram_after = process.memory_info().rss / 1024**3
cpu_ram_used = cpu_ram_after - cpu_ram_before

# Performance Report
print(f"\n" + "="*30)
print(f"   TRAINING PERFORMANCE REPORT")
print(f"="*30)
print(f"Total Training Time:   {train_duration / 60:.2f} minutes")
print(f"Peak VRAM Reserved:    {used_vram} GB ({used_memory_percentage}%)")
print(f"System RAM Consumed:   {cpu_ram_used:.2f} GB")
print(f"Training Speed:        {trainer_stats.metrics['train_samples_per_second']:.2f} samples/sec")
print(f"Final Loss:            {trainer_stats.metrics['train_loss']:.4f}")
print(f"="*30)

In [ ]:
from unsloth.chat_templates import get_chat_template

# Enable 2x faster inference
FastLanguageModel.for_inference(model)

# Define the prompt (Same as training!)
instruction = "Complete the Fibonacci sequence provided in the input."
input_data = "1, 1, 2, 3, 5, 8,"

# Format using the Alpaca template
prompt = alpaca_prompt.format(instruction, input_data, "")

# Tokenize and move to GPU
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

In [ ]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

output = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 64,
    use_cache = True # Required for fast generation
)

In [ ]:
model.save_pretrained("lora_model_unsloth") # Saves the tiny adapter weights
tokenizer.save_pretrained("lora_model_unsloth")